# Messy Data Issues Report  
**Dataset:** `lahore_flats_v2.csv`

---

## 1. Completeness Issues
Significant missing values were found across multiple critical fields:

- **Price / Marla / Area (sqft):** 599 missing each  
- **Floor Number:** 1,503 missing  
- **Total Floors:** 1,468 missing  
- **Built Year:** 1,117 missing  
- **Description:** 599 missing  
- **Features:** 652 missing  
- **Nearby Locations and Other Facilities:** 789 missing  
- **Other Rooms:** 866 missing  
- **Property ID:** Not found for 599 records  
- **Name:** Appears as `#NAME?` in 155 records  

---

## 2. Validity Issues
Presence of outliers and clearly invalid values:

- **Floor Number:** Includes extreme values such as `1234` and `2018`  
- **Total Floors:** Invalid entries like `2018` and `2020`  
- **Built Year:** Contains `1`, `20242`, and boolean `TRUE` (110 rows affected)  
- **Parking Spaces:** Unrealistic values up to `500`  

---

## 3. Accuracy Issues
Type and format inconsistencies affecting data reliability:

- **Price:** Stored as text with currency symbols/units in ~2,500 rows  
- **Floor Number:** Contains boolean values (`TRUE/FALSE`) in 500 rows  
- **Total Floors:** Non-numeric entries in 278 rows  
- **Built Year:** Non-YYYY formats in 78 rows  

---

## 4. Consistency Issues
Conflicting representations and structural inconsistencies:

- **Duplicate `Link` values:** 20 duplicates identified  
- **Boolean vs numeric mixing:**  
  - Columns such as *Parking Spaces*, *Floor*, *Floors in Building*, *Elevators* mix `TRUE/FALSE` with numeric counts  
- **Area mismatch:**  
  - Area (sqft) never aligns with standard Marla-to-sqft conversion  
- **Column naming issue:**  
  - `\ufeffProperty ID` includes a BOM (Byte Order Mark)  

---

## 5. Messy Data Structure Issues

### 5.1 Multi-Valued Cells
- **Features:** Multiple items packed using `|` delimiter  
- **Nearby Locations and Other Facilities:** Comma-separated lists in one cell  
- **Other Rooms:** Comma-separated values stored in a single column  

### 5.2 Structured Data Embedded as Text
- **Rooms:** Stored as a stringified dictionary, packing multiple attributes into one cell  

### 5.3 Sparse Section Columns
- **Section - Plot Features:** Completely empty in 3,098 rows  
- **Other `Section - ...` columns:** High levels of missingness across the dataset  

---

## 6. Summary
The dataset exhibits major issues across **completeness, validity, accuracy, consistency, and structure**.  
Substantial cleaning, normalization, type enforcement, and schema redesign are required before it can be reliably used for analysis or modeling.

---


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [47]:
df=pd.read_csv("E:\Project\Data Cleaning\lahore_flats_v2.csv")

In [3]:
df.sample(10)

,Property ID,Name,Price,Marla,Area (sqft),Floor Number,Total Floors,Built Year,Address,Description,...,Park Facing,Disputed,File,Balloted,Sewerage,Electricity,Water Supply,Sui Gas,Boundary Wall,Section - Plot Features
1534,53450701,"3 BHK Gulberg, Lahore, Punjab",PKR 3.5 Crore,8.1,1822.5,NaN,5,2025,"Gulberg, Lahore, Punjab",luxurious Apartment 2 beds Atched bath maid ro...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
101,53571908,"4 BHK Askari 10 - Sector S, Askari 10, Askari,...",PKR 4.35 Crore,12.0,2700.0,7,9,2025,"Askari 10 - Sector S, Askari 10, Askari, Lahor...",Brand New Good View 4 Bedrooms 4 Baths 1 Powde...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,53506557,#NAME?,PKR 56.52 Lakh,1.4,315.0,9,TRUE,2025,"Midway Commercial, Bahria Town, Lahore, Punjab","Experience luxury living at Grand 11, a presti...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2195,53179414,#NAME?,PKR 1.8 Crore,2.0,450.0,TRUE,TRUE,2029,"Zameen EON, NSIT City, DHA Defence, Lahore, Pu...",Zameen EON Key Highlights Developer: Zameen De...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
684,53474102,"3 BHK Askari 11, Askari, Lahore, Punjab",PKR 3.25 Crore,10.0,2250.0,1,7,2025,"Askari 11, Askari, Lahore, Punjab","Discover a perfect blend of comfort, security,...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
288,53570813,"4 BHK Askari 10 - Sector S, Askari 10, Askari,...",PKR 4.48 Crore,13.0,2925.0,7,9,2025,"Askari 10 - Sector S, Askari 10, Askari, Lahor...",A spacious and brand new 4 bedroom flat availa...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2459,Not found,Not listed,NaN,NaN,NaN,NaN,NaN,NaN,Not listed,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2333,53273223,"2 BHK DHA Phase 8 - Block Q, DHA Phase 8, DHA ...",PKR 2.1 Crore,4.5,1012.5,3,7,2025,"DHA Phase 8 - Block Q, DHA Phase 8, DHA Defenc...",2 Bed Apartment Is For sale In Dha Phase 8 Lah...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
441,53559279,"3 BHK Askari 11 - Sector B, Askari 11, Askari,...",PKR 2.9 Crore,10.0,2250.0,2,8,2025,"Askari 11 - Sector B, Askari 11, Askari, Lahor...","03 Bed Apartment 3Bed 1Kitchen Drawing , Dinin...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
302,53570699,"4 BHK Askari 10, Askari, Lahore, Punjab",PKR 4.45 Crore,12.0,2700.0,7,9,2025,"Askari 10, Askari, Lahore, Punjab",12 Marla 4 Bed Apartment: . 4 Bedrooms . 4 Bat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Analyze issues with data through dynamic & Programetic assessment.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3099 entries, 0 to 3098
Data columns (total 84 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Property ID                                      3099 non-null   object 
 1   Name                                             3099 non-null   object 
 2   Price                                            2500 non-null   object 
 3   Marla                                            2500 non-null   float64
 4   Area (sqft)                                      2500 non-null   float64
 5   Floor Number                                     1596 non-null   object 
 6   Total Floors                                     1631 non-null   object 
 7   Built Year                                       1982 non-null   object 
 8   Address                                          3099 non-null   object 
 9   Description                   

In [134]:
df.describe()

,Marla,Area (sqft)
count,2500.000000,2500.000000
mean,6.248960,1406.016000
std,3.988644,897.444834
min,0.400000,90.000000
25%,2.400000,540.000000
50%,5.000000,1125.000000
75%,10.000000,2250.000000
max,28.500000,6412.500000


## Issues in data

In [5]:
import re
import numpy as np

def price_to_pkr(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    
    s = str(x).strip().lower()
    s = s.replace(",", "")
    
    # common cleanup
    s = re.sub(r"\s+", " ", s)          # normalize spaces
    s = s.replace("pkr", "").strip()    # remove currency label
    
    # extract number + unit
    m = re.search(r"(\d+(?:\.\d+)?)\s*(lakh|crore)", s)
    if not m:
        return np.nan
    
    val = float(m.group(1))
    unit = m.group(2)
    
    if unit == "lakh":
        return val * 100_000
    if unit == "crore":
        return val * 10_000_000
    
    return np.nan

# create numeric column
df["Price"] = df["Price"].apply(price_to_pkr)

# (optional) make it integer rupees where possible
df["Price"] = df["Price"].round().astype("Int64")


In [7]:
df['Price']

0        5400000
1        2157000
2        8000000
3        2000000
4        2500000
          ...   
3094     7500000
3095    12200000
3096     6380000
3097     4500000
3098     5500000
Name: Price, Length: 3099, dtype: Int64

In [8]:
df.columns

Index(['Property ID', 'Name', 'Price', 'Marla', 'Area (sqft)', 'Floor Number',
       'Total Floors', 'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Rooms', 'Other Rooms', 'Link',
       'Parking Spaces', 'Lobby in Building', 'Double Glazed Windows',
       'Central Air Conditioning', 'Central Heating', 'Flooring',
       'Electricity Backup', 'Waste Disposal', 'Floor', 'Floors in Building',
       'Elevators', 'Service Elevators in Building', 'Other Main Features',
       'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Business Center or Media Room in Building',
       'Conference Room in Building', 'Intercom', 'ATM Machines',
       'Other Business and Communication Facilities',
       'Community Lawn or Garden', 'Community Swimming Pool', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre',
       'Other Community Faci

In [9]:
df["Area (sqft)"] = df["Marla"] * 272.25

In [11]:
df["price_per_sqft"] = df["Price"] / df["Area (sqft)"]

In [12]:
df.columns

Index(['Property ID', 'Name', 'Price', 'Marla', 'Area (sqft)', 'Floor Number',
       'Total Floors', 'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Rooms', 'Other Rooms', 'Link',
       'Parking Spaces', 'Lobby in Building', 'Double Glazed Windows',
       'Central Air Conditioning', 'Central Heating', 'Flooring',
       'Electricity Backup', 'Waste Disposal', 'Floor', 'Floors in Building',
       'Elevators', 'Service Elevators in Building', 'Other Main Features',
       'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Business Center or Media Room in Building',
       'Conference Room in Building', 'Intercom', 'ATM Machines',
       'Other Business and Communication Facilities',
       'Community Lawn or Garden', 'Community Swimming Pool', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre',
       'Other Community Faci

In [15]:
df = df[df["Property ID"].str.strip().str.lower() != "not found"]



In [17]:
df.shape

(2500, 85)

In [18]:
df.duplicated().sum()

np.int64(0)

In [101]:
df.columns

Index(['Property ID', 'Name', 'Price', 'Marla', 'Area (sqft)', 'Floor Number',
       'Total Floors', 'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Rooms', 'Other Rooms', 'Link',
       'Parking Spaces', 'Lobby in Building', 'Double Glazed Windows',
       'Central Air Conditioning', 'Central Heating', 'Flooring',
       'Electricity Backup', 'Waste Disposal', 'Floor', 'Floors in Building',
       'Elevators', 'Service Elevators in Building', 'Other Main Features',
       'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Business Center or Media Room in Building',
       'Conference Room in Building', 'Intercom', 'ATM Machines',
       'Other Business and Communication Facilities',
       'Community Lawn or Garden', 'Community Swimming Pool', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre',
       'Other Community Faci

In [96]:
import ast

df['Rooms_dict'] = df['Rooms'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else {}
)


In [ ]:
rooms_df = pd.json_normalize(df['Rooms_dict'])
rooms_df

,Bedrooms,Bathrooms,Kitchens,Servant Quarters,Store Rooms
0,1,1,1,NaN,NaN
1,1,1,1,NaN,NaN
2,1,1,1,NaN,NaN
3,NaN,1,1,1,1
4,1,1,1,1,1
...,...,...,...,...,...
2495,NaN,NaN,NaN,NaN,NaN
2496,1,2,1,NaN,NaN
2497,1,1,NaN,2,1
2498,1,1,1,1,1


In [100]:
df = pd.concat([df, rooms_df], axis=1)

In [83]:
features = df['Features'].apply(lambda x: x.split("|") if isinstance(x, str) else x)

features


0       [Parking Spaces ,  Lobby in Building ,  Double...
1       [Central Air Conditioning ,  Central Heating ,...
2       [Parking Spaces ,  Lobby in Building ,  Double...
3       [Built in year : 2023 ,  Parking Spaces : 1 , ...
4       [Built in year : 2023 ,  Parking Spaces ,  Lob...
                              ...                        
3094    [Built in year : 2020 ,  Parking Spaces : 20 ,...
3095    [Built in year : 2028 ,  Parking Spaces : 3 , ...
3096    [Built in year : 2025 ,  Parking Spaces ,  Lob...
3097    [Built in year : 2022 ,  Parking Spaces : 4 , ...
3098    [Built in year : 2001 ,  Parking Spaces : 40 ,...
Name: Features, Length: 2500, dtype: object

In [104]:
df['Rooms'][90]

"{'Bedrooms': '3', 'Bathrooms': '4', 'Servant Quarters': '1', 'Kitchens': '1', 'Store Rooms': '1'}"

In [107]:
df=df.drop(columns='Rooms')

In [112]:
df['Nearby Locations and Other Facilities'].isna().sum()

np.int64(681)

In [116]:
df.columns

Index(['Property ID', 'Name', 'Price', 'Marla', 'Area (sqft)', 'Floor Number',
       'Total Floors', 'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Other Rooms', 'Link',
       'Parking Spaces', 'Lobby in Building', 'Double Glazed Windows',
       'Central Air Conditioning', 'Central Heating', 'Flooring',
       'Electricity Backup', 'Waste Disposal', 'Floor', 'Floors in Building',
       'Elevators', 'Service Elevators in Building', 'Other Main Features',
       'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Business Center or Media Room in Building',
       'Conference Room in Building', 'Intercom', 'ATM Machines',
       'Other Business and Communication Facilities',
       'Community Lawn or Garden', 'Community Swimming Pool', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre',
       'Other Community Facilities', 

'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Other Rooms', 'Link',
       'Parking Spaces', 'Lobby in Building', 'Double Glazed Windows',
       'Central Air Conditioning', 'Central Heating', 'Flooring',
       'Electricity Backup', 'Waste Disposal', 'Floor', 'Floors in Building',
       'Elevators', 'Service Elevators in Building', 'Other Main Features',
       'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Business Center or Media Room in Building',
       'Conference Room in Building', 'Intercom', 'ATM Machines',
       'Other Business and Communication Facilities',
       'Community Lawn or Garden', 'Community Swimming Pool', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre',
       'Other Community Facilities', 'Sauna', 'Jacuzzi',
       'Other Healthcare and Recreation Facilities', 'Nearby Schools',
       'Nearby Hospitals', 'Nearby Shopping Malls', 'Nearby Restaurants',
       'Distance From Airport (kms)', 'Nearby Public Transport Service',
       'Other Nearby Places', 'Maintenance Staff', 'Security Staff',
       'Laundry or Dry Cleaning Facility', 'Facilities for Disabled',
       'Pets Allowed', 'Other Facilities', 'Section - Main Features',
       'Section - Rooms', 'Section - Business and Communication',
       'Section - Community Features', 'Section - Healthcare Recreational',
       'Section - Nearby Locations and Other Facilities',
       'Section - Other Facilities', 'Furnished', 'Communal/Shared Kitchen',
       'Built in year', 'Pets Not Allowed', 'Possesion', 'Corner',
       'Park Facing', 'Disputed', 'File', 'Balloted', 'Sewerage',
       'Electricity', 'Water Supply', 'Sui Gas', 'Boundary Wall',
       'Section - Plot Features', 'price_per_sqft', 'fe', 'Rooms_dict',
       'Bedrooms', 'Bathrooms', 'Kitchens', 'Servant Quarters', 'Store Rooms'

In [ ]:
df.drop(columns='Property ID')

(2991, 91)

In [127]:
df.loc[df['Built Year'] == 'TRUE', 'Built Year'] = np.nan

ltiple critical fields:

- **Price / Marla / Area (sqft):** 599 missing each  
- **Floor Number:** 1,503 missing  
- **Total Floors:** 1,468 missing  
- **Built Year:** 1,117 missing  
- **Description:** 599 missing  
- **Features:** 652 missing  
- **Nearby Locations and Other Facilities:** 789 missing  
- **Other Rooms:** 866 missing  
- **Property ID:** Not found for 599 records  
- **Name:** Appears as `#NAME?` in 155 records 

## Solve


In [48]:
df.shape

(3099, 84)

In [49]:
df = df.drop_duplicates(subset=["Link"], keep="first")

In [50]:
df = df.dropna(how="all")

In [51]:
df.shape

(2520, 84)

In [52]:
df.columns

Index(['Property ID', 'Name', 'Price', 'Marla', 'Area (sqft)', 'Floor Number',
       'Total Floors', 'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Rooms', 'Other Rooms', 'Link',
       'Parking Spaces', 'Lobby in Building', 'Double Glazed Windows',
       'Central Air Conditioning', 'Central Heating', 'Flooring',
       'Electricity Backup', 'Waste Disposal', 'Floor', 'Floors in Building',
       'Elevators', 'Service Elevators in Building', 'Other Main Features',
       'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Business Center or Media Room in Building',
       'Conference Room in Building', 'Intercom', 'ATM Machines',
       'Other Business and Communication Facilities',
       'Community Lawn or Garden', 'Community Swimming Pool', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre',
       'Other Community Faci

In [53]:
df[df[['Price', 'Marla', 'Area (sqft)']].isna()]

,Property ID,Name,Price,Marla,Area (sqft),Floor Number,Total Floors,Built Year,Address,Description,...,Park Facing,Disputed,File,Balloted,Sewerage,Electricity,Water Supply,Sui Gas,Boundary Wall,Section - Plot Features
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3094,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3095,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3097,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
df = df.dropna(subset=["Price", "Marla", "Area (sqft)"])

In [55]:
df[['Price', 'Marla', 'Area (sqft)']].isna().sum()

Price          0
Marla          0
Area (sqft)    0
dtype: int64

In [56]:
df['Floor Number'].isna().sum()

np.int64(904)

In [57]:
df['Total Floors'].isna().sum()

np.int64(869)

In [58]:
df.loc[df["Floor Number"].isna(), ["Floor Number", "Total Floors"]]


,Floor Number,Total Floors
1,NaN,13
10,NaN,TRUE
15,NaN,NaN
16,NaN,NaN
22,NaN,NaN
...,...,...
3087,NaN,NaN
3089,NaN,NaN
3091,NaN,NaN
3092,NaN,NaN


In [59]:
df.shape

(2500, 84)

In [60]:
df["Total Floors"] = pd.to_numeric(df["Total Floors"], errors="coerce")
df["Floor Number"] = pd.to_numeric(df["Floor Number"], errors="coerce")

In [61]:
df["Total Floors"] = df["Total Floors"].fillna(
    df.groupby("Address")["Total Floors"].transform("median")
)

In [62]:
df["Total Floors"] = df["Total Floors"].fillna(df["Total Floors"].median())

In [63]:
df["Floor Number"] = df["Floor Number"].fillna(
    df.groupby("Address")["Floor Number"].transform("median")
)

df["Floor Number"] = df["Floor Number"].fillna(df["Floor Number"].median())

In [64]:
df.loc[df["Floor Number"].isna(), ["Floor Number", "Total Floors"]]


,Floor Number,Total Floors


In [65]:
df['Built Year'].isna().sum()

np.int64(518)

In [136]:
df['Built Year'].value_counts()

Built Year
2025.0    613
2024.0    272
2023.0    115
2026.0     98
2022.0     94
2027.0     78
2018.0     75
2020.0     69
2021.0     68
2017.0     56
2029.0     49
2028.0     46
2015.0     33
2019.0     29
1.0        24
2016.0     18
2.0        14
2010.0     12
2030.0     12
2012.0     11
3.0        10
2005.0      9
2000.0      7
4.0         6
1992.0      6
2014.0      5
2008.0      4
1998.0      4
2003.0      4
1985.0      3
2001.0      3
2011.0      3
1986.0      2
1999.0      2
2004.0      2
1990.0      2
2002.0      2
1994.0      1
1987.0      1
1996.0      1
2006.0      1
2007.0      1
Name: count, dtype: int64

In [ ]:
df["Built Year"] = pd.to_numeric(df["Built Year"], errors="coerce")

In [141]:
df.loc[df["Built Year"] == 1 , "Built Year"] = 2025

In [142]:
df[df['Built Year']==2]['Link'].iloc[4]

'https://www.zameen.com/Property/gulberg_gulberg_3_1_bed_room_apartment_for_sale_in_gulberg_lahore-52799035-3824-1.html'

In [117]:
df['Built Year'].value_counts()

Built Year
2025.0     613
2024.0     272
2023.0     115
2026.0      98
2022.0      94
2027.0      78
2018.0      75
2020.0      69
2021.0      67
2017.0      56
2029.0      49
2028.0      46
2015.0      33
2019.0      29
1.0         24
2016.0      18
2.0         14
2030.0      12
2010.0      11
2012.0      11
3.0         10
2000.0       7
4.0          6
1992.0       6
2014.0       5
2005.0       5
5.0          4
2003.0       4
2008.0       4
1998.0       4
1985.0       3
2011.0       3
2001.0       2
2002.0       2
1986.0       2
1999.0       2
1990.0       2
20242.0      2
2004.0       2
20221.0      1
1994.0       1
1987.0       1
20001.0      1
2056.0       1
1996.0       1
10.0         1
6.0          1
2007.0       1
Name: count, dtype: int64

In [ ]:
df["Built Year"] = pd.to_numeric(df["Built Year"], errors="coerce")
df.loc[~df["Built Year"].between(1980, 2030), "Built Year"] = np.nan
